# Loan Approval Predictor

1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

2. Load Dataset

In [ ]:
df = pd.read_csv("loan_applicants.csv")

3. Gather basic information

In [ ]:
df.head()
df.info()
df.isnull().sum()

In [ ]:
cat_col = df.select_dtypes(include=["object","str"]).columns
num_col = df.select_dtypes(include=["number"]).columns

In [ ]:
# Handling Missing Data
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="mean")
df[num_col] = num_imp.fit_transform(df[num_col])

col_imp = SimpleImputer(strategy="most_frequent")
df[cat_col] = col_imp.fit_transform(df[cat_col])

4. Exploratory Data Analysis(EDA)

In [ ]:
# Visulaizing Data
# Number of loan approved

classes_count = df["Loan_Approved"].value_counts()

plt.pie(classes_count, labels=["No","Yes"], autopct="%1.1f%%")
plt.title("Is Loan Approved or not ?")

In [ ]:
# Relationship betweeen Gender, Education level and loan approval

gender_cat = df["Gender"].value_counts()
ax = sns.barplot(gender_cat)
ax.bar_label(ax.containers[0])

edu_cat = df["Education_Level"].value_counts()
ax = sns.barplot(edu_cat)
ax.bar_label(ax.containers[0])

sns.histplot(
    data = df,
    x = "Applicant_Income",
    bins = 20
 )

sns.histplot(
    data = df,
    x = "Coapplicant_Income",
    bins = 20
)

sns.boxplot(
    data = df,
    x = "Loan_Approved",
    y = "Applicant_Income"
)

In [ ]:
# Relationship betweeen credit score and loan approval

sns.histplot(
    data=df,
    x = "Credit_Score",
    hue = "Loan_Approved",
    bins = 20,
    multiple = "dodge"
)

In [ ]:
# Relationship across other features with respect to loan approved

fig,axes = plt.subplots(2,2)
sns.boxplot(ax=axes[0,0], data = df, x = "Loan_Approved", y = "Applicant_Income")
sns.boxplot(ax=axes[0,1], data = df, x = "Loan_Approved", y = "Credit_Score")
sns.boxplot(ax=axes[1,0], data = df, x = "Loan_Approved", y = "DTI_Ratio")
sns.boxplot(ax=axes[1,1], data = df, x = "Loan_Approved", y = "Savings")

plt.tight_layout()

In [ ]:
# Remove Unecessary Features
df = df.drop("Applicant_ID", axis=1)

In [ ]:
# Encoding
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

# Label Encoding
le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])

# One HotEncoding
cols = ["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]
ohe = OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns= ohe.get_feature_names_out(cols),index=df.index)
df = pd.concat([df.drop(columns=cols),encoded_df], axis=1)

In [ ]:
#Corelation HeatMap
num_cols = df.select_dtypes(include="number")
corr_mat = num_cols.corr()
# num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

plt.figure(figsize=(21,8))
sns.heatmap(
    corr_mat,
    annot=True,
    fmt='.2f',
    cmap="coolwarm"
)
plt.tight_layout()

5. Split data for training and testing

In [ ]:
X = df.drop("Loan_Approved",axis=1)
y = df["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

6. Scale data using StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

7. Train and Evaluate the models

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score,confusion_matrix,accuracy_score,recall_score,f1_score,classification_report

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)
y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision:", precision_score(y_test,y_pred))
print("Recall:", recall_score(y_test,y_pred))
print("F1 Score:", f1_score(y_test,y_pred))
print("Accuracy:", accuracy_score(y_test,y_pred))
print("Confusion Matrix:", confusion_matrix(y_test,y_pred))

In [ ]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

gnb_model = GaussianNB()
gnb_model.fit(X_train_scaled, y_train)
y_pred = gnb_model.predict(X_test_scaled)

# Evaluation
print(" Naive Bayes Model")
print("Precision:", precision_score(y_test,y_pred))
print("Recall:", recall_score(y_test,y_pred))
print("F1 Score:", f1_score(y_test,y_pred))
print("Accuracy:", accuracy_score(y_test,y_pred))
print("Confusion Matrix:", confusion_matrix(y_test,y_pred))

In [ ]:
# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=13)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Classifier Model")
print("Precision:", precision_score(y_test,y_pred))
print("Recall:", recall_score(y_test,y_pred))
print("F1 Score:", f1_score(y_test,y_pred))
print("Accuracy:", accuracy_score(y_test,y_pred))
print("Confusion Matrix:", confusion_matrix(y_test,y_pred))

In [ ]:
# XGBOOST with hyperparameter tuning
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    eval_metric="auc",
    random_state=42
)
xgb_model.fit(X_train_scaled,y_train)
y_pred = xgb_model.predict(X_test_scaled)

# Evaluation
print("XGBoost Classifier Model")
print("Accuracy: ",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

8. Save models

In [ ]:
import joblib

joblib.dump(log_model,"logistic_model.pkl")
joblib.dump(gnb_model,"NaiveBayes_model.pkl")
joblib.dump(knn_model,"knn_model.pkl")
joblib.dump(xgb_model,"xgboost_model.pkl")